# TMAP Quickstart

TMAP produces a **tree-structured 2D map** of your data. Unlike UMAP or t-SNE
(which give you a cloud of disconnected points), TMAP connects every point
via a minimum spanning tree — so you can trace the path between any two items.

This notebook covers the high-level API. Three lines to a working TMAP:

```python
from tmap import TMAP
model = TMAP().fit(data)
model.plot()  # or model.to_html("output.html")
```

## 1. Create sample data

We generate synthetic binary fingerprints with 3 clusters.
In practice you would use molecular fingerprints, protein embeddings, etc.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
n_samples, n_bits, n_clusters = 5000, 2048, 3

fingerprints = np.zeros((n_samples, n_bits), dtype=np.uint8)
cluster_labels = []

for i in range(n_samples):
    c = i % n_clusters
    cluster_labels.append(f"Cluster {c}")
    # Each cluster has a distinct region of active bits
    base = rng.choice(400, 50, replace=False) + c * 500
    fingerprints[i, base] = 1
    fingerprints[i, rng.choice(n_bits, 20, replace=False)] = 1

print(f"{n_samples} fingerprints, {n_bits} bits, avg set: {fingerprints.sum(1).mean():.0f}")


## 2. Fit TMAP

The `TMAP` class follows the sklearn pattern. Default metric is `'jaccard'`
(binary fingerprints via MinHash + LSH Forest). For dense embeddings, see
the [Metric Guide](06_metric_guide.ipynb).

In [ ]:
from tmap import TMAP

model = TMAP(n_neighbors=20, seed=42).fit(fingerprints)

print(f"Embedding:  {model.embedding_.shape}")
print(f"Tree edges: {model.tree_.edges.shape[0]}")
print(f"kNN graph:  {model.graph_.indices.shape}")


## 3. Visualize

### Jupyter notebook (interactive)

In [ ]:
# Quick interactive plot — pass any array or DataFrame column as color
model.plot(color_by=cluster_labels)


### HTML export (shareable)

`TmapViz` gives full control: multiple color layers, labels, edge styling,
SMILES rendering.

In [ ]:
viz = model.to_tmapviz()
viz.title = "Quickstart Demo"
viz.background_color = "#FFFFFF"

# Add color layers (switch between them in the dropdown)
viz.add_color_layout("Cluster", cluster_labels, categorical=True, color="Set1")

values = np.random.default_rng(0).uniform(0, 100, n_samples)
viz.add_color_layout("Random value", values.tolist(), categorical=False, color="viridis")

# Tooltip labels
viz.add_label("ID", [f"Point {i}" for i in range(n_samples)])

# Export self-contained HTML (opens in any browser)
path = viz.write_html("./")
print(f"Saved: {path}")


### Matplotlib (static, for papers)

In [ ]:
model.plot_static(color_by=cluster_labels, color_map="Set1", point_size=2)


## 4. Explore the tree

This is what makes TMAP different from UMAP/t-SNE. The tree gives you
**paths** between any two points and **distances** along those paths.

In [ ]:
# Unique path between two points
path = model.path(0, 100)
print(f"Path from 0 to 100: {len(path)} steps")
print(f"  First 10 nodes: {path[:10]}")

# Tree distance (sum of edge weights along the path)
d = model.distance(0, 100)
print(f"  Tree distance: {d:.2f}")

# "Pseudotime": distance from a root to all other points
distances = model.distances_from(0)
print(f"\nDistances from node 0: min={distances.min():.2f}, max={distances.max():.2f}")

# Color by pseudotime
model.plot(color_by=distances, color_map="magma")


## 5. Save and reload

In [ ]:
model.save("my_tmap.pkl")
loaded = TMAP.load("my_tmap.pkl")
print(f"Loaded model: {loaded.embedding_.shape}")
print(f"Same embedding: {np.allclose(model.embedding_, loaded.embedding_)}")


## Summary

| What | How |
|------|-----|
| Fit | `model = TMAP(n_neighbors=20).fit(data)` |
| Coordinates | `model.embedding_` |
| Tree topology | `model.tree_` |
| Path between points | `model.path(a, b)` |
| Pseudotime | `model.distances_from(root)` |
| Jupyter plot | `model.plot(color_by=...)` |
| Static plot | `model.plot_static(color_by=...)` |
| HTML export | `model.to_html("output.html")` |
| Save/load | `model.save(path)` / `TMAP.load(path)` |

## Next steps

- [Visualization Guide](10_visualization_guide.ipynb) — SMILES rendering, image thumbnails, protein 3D viewer, filtering, cards
- [Metric Guide](06_metric_guide.ipynb) — cosine, euclidean, precomputed, external kNN
- [FAQ & Troubleshooting](07_faq.ipynb) — disconnected nodes, compact layouts, tuning
- [Cheminformatics](08_cheminformatics.ipynb) — full drug-discovery workflow with `add_smiles`
- [Protein Analysis](09_protein_analysis.ipynb) — FASTA, embeddings, AlphaFold, UniProt
- [MinHash Deep Dive](02_minhash_deep_dive.ipynb) — encoding details for binary data
- [Parameter Tuning](03_tuning_parameters.ipynb) — LayoutConfig options
- [Jupyter Interaction](04_jscatter_demo.ipynb) — TmapViz, selection, DataFrames